# Notebook Overview
This notebook is the **noRerank** ablation variant of the engineering RAG system. Compared with the full pipeline, this version **removes the LLM-based semantic re-ranking stage** — after vector retrieval, the top-K chunks are selected purely by cosine similarity, without GPT‑4o re-ranking by physical-condition fit.

The core workflow retains the full agent loop (Planner → Retriever → Solver → Critic → back), so the system can still iterate and self-correct. However, the Retriever's answer quality relies solely on embedding similarity, making it possible to isolate the contribution of **semantic re-ranking** to overall system performance.

**Key difference from the full pipeline:**
- In the full pipeline, the Retriever fetches candidate chunks by embedding similarity, then uses GPT‑4o to re-rank them by how well their physical conditions match the sub-problem.
- In this variant, the re-ranking pass is skipped — chunks are ranked purely by cosine similarity against the task description embedding.

**Workflow summary:**
1. **Knowledge preparation** — Load JSONL chunks, inspect schema, clean formula text.
2. **Question decomposition & translation** — Translate CN question → EN, then decompose into sub-problems via DeepSeek-V3.
3. **Vector retrieval** — Generate Qwen3-Embedding vectors, retrieve top-K chunks by cosine similarity (no re-rank).
4. **Agent loop** — Planner → Retriever (no re-rank) → Solver → Critic → iterate up to 3 rounds.
5. **Batch execution** — Run the full pipeline across a range of questions.

---

本 notebook 是工程 RAG 系统的 **noRerank** 消融变体。与完整流程相比，该版本**移除了基于 LLM 的语义重排序阶段** —— 向量检索后，Top-K 知识块直接按余弦相似度选取，不再经过 GPT‑4o 按物理条件匹配度进行重排。

核心流程保留了完整的 Agent 迭代闭环（Planner → Retriever → Solver → Critic → 回退），因此系统仍可自我纠错。但 Retriever 的输出质量仅依赖于向量相似度，从而可以量化**语义重排序**对系统性能的独立贡献。

**与完整流程的关键差异：**
- 完整流程中，Retriever 先通过向量相似度获取候选集，再用 GPT‑4o 按物理条件匹配度进行重排。
- 本变体中，重排序步骤被跳过 —— 知识块仅按与 Task Description 的余弦相似度排序。


In [1]:
import os
from config import SILICONFLOW_API_KEY, SILICONFLOW_BASE_URL

os.environ["OPENAI_API_KEY"] = SILICONFLOW_API_KEY
os.environ["OPENAI_BASE_URL"] = SILICONFLOW_BASE_URL

### 1. Knowledge preparation
This section loads JSONL chunks, inspects the schema, and normalizes formula text.

---

这部分负责加载 JSONL 知识块、查看数据结构，并清洗公式文本。


In [2]:
import os
import json
from typing import List, Dict

def load_jsonl_files(folder_path: str) -> List[Dict]:
    """
    加载文件夹中所有JSONL文件，返回所有chunk的列表
    :param folder_path: JSONL文件所在文件夹路径
    :return: 包含所有chunk的列表，每个元素是一个chunk字典
    """
    chunks = []
    # 遍历文件夹中的所有文件
    if not os.path.exists(folder_path):
        print(f"⚠️ 警告: 文件夹 {folder_path} 不存在")
        return []
        
    for filename in os.listdir(folder_path):
        if filename.endswith(".jsonl"):  # 只处理JSONL文件
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:  # 跳过空行
                        try:
                            chunk = json.loads(line)  # 解析JSON行
                            chunks.append(chunk)
                        except json.JSONDecodeError:
                            continue
    print(f"成功加载 {len(chunks)} 个chunk")
    return chunks

# 示例调用：加载EEjsonl文件夹中的所有chunk
folder_path = "jsons_EE_Eng"  # 你的JSONL文件夹路径
all_chunks = load_jsonl_files(folder_path)

成功加载 539 个chunk


In [3]:
# 检查chunk的实际结构
print("📊 检查chunk结构...")
if all_chunks:
    sample_chunk = all_chunks[0]
    print(f"第一个chunk的键: {list(sample_chunk.keys())}")
    print(f"第一个chunk的内容示例:")
    for key, value in sample_chunk.items():
        if isinstance(value, str):
            print(f"  {key}: {value[:100]}...")
        elif isinstance(value, list):
            print(f"  {key}: {value[:3]}...")
        else:
            print(f"  {key}: {value}")
    print("\n" + "="*50)

📊 检查chunk结构...
第一个chunk的键: ['chunk_id', 'formula', 'full_context', 'variable_definitions', 'keywords', 'potential_questions', 'task_description', 'section', 'difficulty_level', 'discipline']
第一个chunk的内容示例:
  chunk_id: DL_T_5044-2014-A.3.1-001...
  formula: I_{\mathrm{n}} \geq K_{\mathrm{k}} I_{\mathrm{m}}...
  full_context: The rated current of the circuit breaker in the output circuit of the charger should be selected acc...
  variable_definitions: [{'symbol': 'I_n', 'name': 'Circuit breaker rated current', 'meaning': 'Rated current of the circuit protective device', 'unit': 'A'}, {'symbol': 'K_k', 'name': 'Reliability factor', 'meaning': 'Safety factor considering calculation error and reliability', 'unit': 'Dimensionless (take 1.2)'}, {'symbol': 'I_m', 'name': 'Charger rated output current', 'meaning': 'Maximum continuous operating current output by the charger', 'unit': 'A'}]...
  keywords: ['circuit breaker', 'rated current', 'charger']...
  potential_questions: ['Given charger ra

In [4]:
# 文本清理与日志工具
import re
from typing import Any

def _strip_latex_markdown(text: Any) -> str:
    r"""
    清理文本中的 LaTeX/Markdown 痕迹与零宽字符，尽量保留语义内容。
    - 去除：$, $$, \( \), \[ \], \left \right 等定界符
    - 展平：\text{...}、\mathrm{...} 等命令，保留花括号内容
    - 替换：\times -> ×
    - 清理：零宽字符与多余反斜杠
    - 清理：Markdown 加粗/斜体标记 (**...**, __...__)
    - 保留：希腊字母（转为英文拼写或直接去反斜杠）
    """
    if not isinstance(text, str):
        return str(text)
    s = text
    # 去零宽字符
    s = re.sub(r"[\u200b\u200c\u200d]", "", s)
    # 去 Markdown 加粗/斜体
    s = re.sub(r"\*\*|__", "", s)
    # 去美元符与行间公式定界
    s = re.sub(r"\$\$?", "", s)
    # 去行内/行间公式括号
    s = re.sub(r"\\\(|\\\)|\\\[|\\\]", "", s)
    # 去 \left \right 等尺寸控制
    s = re.sub(r"\\left|\\right", "", s)
    
    # 展平 \text{...} / \mathrm{...} / \mathbf{...} / \mathit{...}
    s = re.sub(r"\\text\{([^}]*)\}", r"\1", s)
    s = re.sub(r"\\mathrm\{([^}]*)\}", r"\1", s)
    s = re.sub(r"\\mathbf\{([^}]*)\}", r"\1", s)
    s = re.sub(r"\\mathit\{([^}]*)\}", r"\1", s)
    
    # 常见符号替换
    s = re.sub(r"\\times", "×", s)
    s = re.sub(r"\\cdot", "·", s)
    s = re.sub(r"\\approx", "≈", s)
    s = re.sub(r"\\leq", "≤", s)
    s = re.sub(r"\\geq", "≥", s)
    
    # 希腊字母映射（保留语义）
    greek_map = {
        r"\\sigma": "sigma", r"\\theta": "theta", r"\\alpha": "alpha", 
        r"\\beta": "beta", r"\\gamma": "gamma", r"\\delta": "delta",
        r"\\rho": "rho", r"\\lambda": "lambda", r"\\mu": "mu",
        r"\\pi": "pi", r"\\omega": "omega", r"\\epsilon": "epsilon",
        r"\\Delta": "Delta"
    }
    for k, v in greek_map.items():
        s = re.sub(k, v, s)

    # 剩余命令名：保留单词，去掉反斜杠 (例如 \frac -> frac, \sqrt -> sqrt)
    # 这样至少保留了语义，而不是直接删除
    s = re.sub(r"\\([a-zA-Z]+)", r" \1 ", s)
    
    # 多余空白合并
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"[\n\r]+", "\n", s)
    
    return s.strip()

# 相似度匹配日志缓冲区（避免Notebook折叠后看不到详细过程）
try:
    _similarity_log_buffer  # type: ignore[name-defined]
except NameError:
    _similarity_log_buffer = []

### 2. Question decomposition & translation
This step first translates the Chinese question into English, then decomposes it into sub-problems before retrieval starts — as is standard in the full pipeline. In this noRerank variant, the decomposition step is unchanged; only the later re-ranking is removed.

---

这一步先将中文题目翻译为英文，再拆分为子问题——与完整流程一致。在 noRerank 变体中，拆解步骤不变，仅后续的重排序环节被移除。


In [5]:
## DeepSeek-V3 API配置和复合题目拆解（英文翻译+拆解版）
from openai import OpenAI
import os
import re
from typing import List, Dict, Tuple

# 配置硅基流动 DeepSeek-V3 API客户端
from config import SILICONFLOW_API_KEY, SILICONFLOW_BASE_URL

_api_key = SILICONFLOW_API_KEY
_base_url = SILICONFLOW_BASE_URL
client = OpenAI(api_key=_api_key, base_url=_base_url)

def translate_to_english(text: str) -> str:
    """将中文文本翻译为英文"""
    try:
        response = client.chat.completions.create(
            model="deepseek-ai/DeepSeek-V3",
            messages=[
                {"role": "system", "content": "You are a professional translator specializing in Engineering Field. Translate the following problem from Chinese to English. Ensure high accuracy for all technical terms, parameter names, and numerical values. Do not summarize or omit any information. Only output the translated text."},
                {"role": "user", "content": text}
            ],
            temperature=0.1
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ 翻译失败: {e}")
        return text

def extract_sub_problems(text: str) -> List[str]:
    """从拆解文本中提取子问题列表，适配英文格式。"""
    if not isinstance(text, str) or not text.strip():
        return []
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    subs: List[str] = []
    
    current = []
    found_first = False
    
    # 增强的正则：
    # 1. 允许 Sub-problem, Subproblem, Sub problem, Step
    # 2. 允许前面有任意非字母字符或数字 (比如 bullets, brackets, 序号 1.)
    # Pattern: Start + [Non-Alpha]* + (Keyword) + Space* + Digit
    header_pattern = r"^[^a-zA-Z]*(Sub[- ]?problem|Step)\s*\d+"
    
    for ln in lines:
        match = re.match(header_pattern, ln, re.IGNORECASE)
        
        if match:
            # 如果之前已经在累积一个子问题，先保存
            if found_first and current:
                full_sub = " ".join(current).strip()
                subs.append(full_sub)
                current = []
            
            found_first = True
            # 清理标题
            # 策略：找到 match 的结束位置，截取后面的部分，并去掉紧随其后的标点
            match_end = match.end()
            remainder = ln[match_end:]
            # 去掉开头的标点符号 (如 ] : . )
            ln_clean = re.sub(r"^[\W_]+", "", remainder).strip()
            
            if ln_clean:
                current.append(ln_clean)
        else:
            # 只有当已经找到第一个子问题后，才开始累积内容
            if found_first:
                current.append(ln)
                
    # 保存最后一个
    if found_first and current:
        full_sub = " ".join(current).strip()
        subs.append(full_sub)
        
    # 2) 若未提取到，尝试按“1.”或“1)”编号行分段 (Fallback)
    if not subs:
        print("⚠️ 未匹配到标准标题，尝试使用数字编号分段...")
        buf = []
        for ln in lines:
            if re.match(r"^\d+[\.\)]", ln):
                if buf:
                    subs.append(" ".join(buf).strip())
                    buf = []
                buf.append(re.sub(r"^\d+[\.\)]\s*", "", ln))
            else:
                if buf:
                    buf.append(ln)
        if buf:
            subs.append(" ".join(buf).strip())
            
    # 3) 仍为空则返回整段作为单一子问题
    if not subs and text.strip():
        print("⚠️ 无法分段，返回整段内容")
        subs = [text.strip()]
        
    return subs

def decompose_complex_problem(user_question: str) -> Tuple[List[str], str]:
    """
    1. 将中文问题翻译为英文
    2. 使用英文Prompt拆解为英文子问题
    返回: (子问题列表, 英文翻译后的问题)
    """
    
    # 1. 翻译
    print("🌐 正在将问题翻译为英文...")
    english_question = translate_to_english(user_question)
    print(f"📝 英文问题: {english_question}")

    # 2. 英文拆解Prompt
    decompose_prompt = f"""
You are a rigorous Engineering Field Problem Decomposition Expert. Your task is to break down a complex problem into a series of logically tight, sequentially correct sub-problems. Each sub-problem must be a minimal solvable unit, and the input/output of consecutive steps must be coherent.

【Decomposition Principles】
1. Logic Chain First: Start from the target variable and reverse engineer the dependency chain.
2. Minimality & Coherence: Each sub-problem solves only one core calculation (one formula). If a variable needed in a formula cannot be obtained directly from the question, it implies a preceding sub-problem is needed.
3. Domain Specifics: Accurately identify special processes like temperature correction, salt density calculation, etc.
4. Variable Steps: Automatically determine the number of steps (2, 3, 4 or more) based on the actual variable dependency.
5. LaTeX Format: All variables and formulas must be in LaTeX format (e.g., use \\sigma_\\theta instead of σ_θ, \\times instead of ×).

【Example A: Engineering Field (3 Steps)】

Problem: "Maintenance personnel conduct a washing test on a polluted insulator to assess its pollution level. The measured volume conductivity of the washing liquid at temperature \\theta = 28^\\circ C is \\sigma_\\theta = 0.15 \\text{{S/m}}, with a temperature coefficient b = 0.032/^\\circ C. The washing used V = 800 \\text{{cm}}^3 of distilled water, and the effective surface area of the insulator is A = 1200 \\text{{cm}}^2. Calculate the Equivalent Salt Deposit Density (ESDD)."

Decomposition Logic:
- First, use the conductivity temperature correction formula to correct the measured conductivity to the standard temperature 20^\\circ C (\\sigma_{{20}}).
- Then, use the corrected conductivity \\sigma_{{20}} to calculate salinity S_s.
- Finally, use salinity S_s and washing parameters to calculate ESDD.

Output:

[Sub-problem 1] Given measured conductivity \\sigma_\\theta = 0.15 \\text{{S/m}}, temperature coefficient b = 0.032/^\\circ C, temperature \\theta = 28^\\circ C, calculate conductivity at 20^\\circ C \\sigma_{{20}} using formula \\sigma_{{20}} = \\sigma_\\theta \\times [1 - b(\\theta - 20)].
→ Input: \\sigma_\\theta, b, \\theta | Formula: \\sigma_{{20}} = \\sigma_\\theta \\times [1 - b(\\theta - 20)] | Output: \\sigma_{{20}}

[Sub-problem 2] Given \\sigma_{{20}}, calculate salinity S_s using formula S_s = (5.7 \\times \\sigma_{{20}})^{{1.03}}.
→ Input: \\sigma_{{20}} | Formula: S_s = (5.7 \\times \\sigma_{{20}})^{{1.03}} | Output: S_s

[Sub-problem 3] Given S_s, V = 800 \\text{{cm}}^3, A = 1200 \\text{{cm}}^2, calculate ESDD using formula \\text{{ESDD}} = S_s \\times V / A.
→ Input: S_s, V, A | Formula: \\text{{ESDD}} = S_s \\times V / A | Output: \\text{{ESDD}}

【Example B: Conductor Temperature Rise & Loss (2 Steps)】

Problem: "A copper conductor has resistance R_{{20}} = 0.50 \\Omega at 20^\\circ C, temperature coefficient \\alpha = 0.004/^\\circ C, operating current I = 50 \\text{{A}}, operating temperature \\theta = 60^\\circ C. Calculate power loss P_{{\\text{{loss}}}} at operating temperature."

Output:
[Sub-problem 1] Calculate resistance R_\\theta at operating temperature using temperature correction formula R_\\theta = R_{{20}} \\times [1 + \\alpha(\\theta - 20)].
→ Input: R_{{20}}, \\alpha, \\theta | Formula: R_\\theta = R_{{20}} \\times [1 + \\alpha(\\theta - 20)] | Output: R_\\theta

[Sub-problem 2] Calculate power loss P_{{\\text{{loss}}}} using loss formula P_{{\\text{{loss}}}} = I^2 \\times R_\\theta.
→ Input: I, R_\\theta | Formula: P_{{\\text{{loss}}}} = I^2 \\times R_\\theta | Output: P_{{\\text{{loss}}}}

【Example C: Three-phase Line Drop & Loss (4 Steps)】

Problem: "Three-phase symmetric supply system, rated line voltage U_N = 10 \\text{{kV}}, line length L = 5 \\text{{km}}, resistance per unit length r = 0.3 \\Omega/\\text{{km}}, reactance per unit length x = 0.2 \\Omega/\\text{{km}}, load current I = 120 \\text{{A}}, power factor \\cos\\phi = 0.85 (lagging). Calculate line voltage drop and active power loss."

Output:
[Sub-problem 1] Calculate line total resistance R = r \\times L.
→ Input: r, L | Formula: R = r \\times L | Output: R

[Sub-problem 2] Calculate line total reactance X = x \\times L.
→ Input: x, L | Formula: X = x \\times L | Output: X

[Sub-problem 3] Calculate line voltage drop \\Delta U = \\sqrt{{3}} \\times I \\times (R \\times \\cos\\phi + X \\times \\sin\\phi).
→ Input: I, R, X, \\cos\\phi | Formula: \\Delta U = \\sqrt{{3}} \\times I \\times (R \\times \\cos\\phi + X \\times \\sin\\phi) | Output: \\Delta U

[Sub-problem 4] Calculate active power loss P_{{\\text{{loss}}}} = 3 \\times I^2 \\times R.
→ Input: I, R | Formula: P_{{\\text{{loss}}}} = 3 \\times I^2 \\times R | Output: P_{{\\text{{loss}}}}

【Your Task】
Strictly follow the decomposition logic above to build the optimal solution path for the following problem:

【Problem Statement】
{english_question}

【Output Format】

【Variable Extraction】
Known Variables: [List all knowns and values]
Target Variable: [Final required variable]

【Formula Path Analysis】
Step-by-step list of formulas, inputs, and outputs.

【Sub-problem Decomposition】
[Sub-problem N] [Detailed description of step N, including specific values and formula name]
→ Input: [Variable Names] | Formula: [Specific Formula] | Output: [Result Variable]

【Formatting Requirements】
- Use LaTeX format for all variables and formulas (e.g., \\sigma_\\theta, \\times, \\sqrt{{...}}).
- Do NOT use Markdown delimiters like $ or $$ for inline math, just write the LaTeX code directly.
- Output must be clear and parsable.
"""

    try:
        print("🔧 正在调用DeepSeek-V3拆解复合题目(英文版)...")
        response = client.chat.completions.create(
            model="deepseek-ai/DeepSeek-V3",
            messages=[
                {"role": "system", "content": "You are an expert in analyzing Engineering Field problems."},
                {"role": "user", "content": decompose_prompt}
            ],
            temperature=0.1,
            max_tokens=2000
        )
        decomposed_text = response.choices[0].message.content or ""
        decomposed_text_clean = _strip_latex_markdown(decomposed_text)
        print("✅ DeepSeek-V3拆解完成")
        print(f"📋 拆解结果（原始）：\n{decomposed_text}")
        
        sub_problems = extract_sub_problems(decomposed_text_clean)
        print(f"🔍 提取到 {len(sub_problems)} 个子问题")
        return sub_problems, english_question
    except Exception as e:
        print(f"❌ DeepSeek-V3 API调用失败: {str(e)}")
        print("🔄 回退到原始问题处理")
        return ([english_question] if english_question.strip() else []), english_question

### 3. Retrieval and prompt construction
This section builds embeddings, retrieves candidate chunks by cosine similarity, and prepares the prompt material. **Key ablation**: unlike the full pipeline, there is no LLM-based semantic re-ranking here — the top-K chunks are used directly based on vector similarity alone.

---

这部分负责生成向量、按余弦相似度检索候选知识块，并整理后续提示词所需内容。**关键消融点**：与完整流程不同，此处没有基于 LLM 的语义重排序——Top-K 知识块直接按向量相似度选取，不再经过 GPT 重排。


In [6]:
# Embedding 组件与缓存工具
import torch
from transformers import AutoTokenizer, AutoModel
import json
import gc
import os
from pathlib import Path
from tqdm import tqdm
from typing import List, Dict, Tuple

# 缓存配置
EMB_CACHE_DIR = "emb_cache_eng"
EMB_CACHE_FILE = os.path.join(EMB_CACHE_DIR, "chunks_embeddings.pt")
EMB_META_FILE = os.path.join(EMB_CACHE_DIR, "chunks_meta.json")

# 全局缓存变量
_cached_chunk_embs = None
_cached_chunk_ids = None

def load_embedding_components(model_path: str = "Qwen3-Embedding") -> Tuple[AutoModel, AutoTokenizer]:
    """加载Qwen3-Embedding模型与分词器。"""
    # 强制先清理显存
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"Loading embedding model from {model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    # CPU: device_map="cpu"
    model = AutoModel.from_pretrained(model_path, trust_remote_code=True, device_map="cpu")
    model.eval()
    return model, tokenizer

def get_embeddings(texts: List[str], model, tokenizer, device: str = None) -> torch.Tensor:
    """对文本列表生成L2归一化的句向量。"""
    # 强制CPU优先
    if device is None:
        device = "cpu"
    
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        out = model(**inputs)
        emb = out.last_hidden_state.mean(dim=1)
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    
    return emb.cpu()

def precompute_chunk_embeddings(chunks: List[Dict], model, tokenizer) -> None:
    """为知识库预计算并缓存向量（强制CPU分批处理以避免OOM）。"""
    global _cached_chunk_embs, _cached_chunk_ids
    print("🚀 Precomputing embeddings (Force CPU)...")
    
    Path(EMB_CACHE_DIR).mkdir(parents=True, exist_ok=True)
    texts, chunk_ids = [], []
    for idx, ch in enumerate(chunks):
        if not isinstance(ch, dict): continue
        desc = ch.get("task_description") or ch.get("formula") or ch.get("full_context") or f"chunk_{idx}"
        try:
             clean_desc = _strip_latex_markdown(str(desc))
        except NameError:
             clean_desc = str(desc).strip()
        texts.append(clean_desc)
        chunk_ids.append(str(ch.get("chunk_id", f"idx_{idx}")))
    
    if not texts:
        _cached_chunk_embs, _cached_chunk_ids = None, None
        return

    batch_size = 32
    all_embs = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        emb = get_embeddings(batch, model, tokenizer, device="cpu")
        all_embs.append(emb)
    
    if all_embs:
        embs = torch.cat(all_embs, dim=0)
        _cached_chunk_embs = embs
        _cached_chunk_ids = chunk_ids
        print(f"✅ Saved to {EMB_CACHE_FILE}")
        torch.save(embs, EMB_CACHE_FILE)
        with open(EMB_META_FILE, "w", encoding="utf-8") as f:
            json.dump(chunk_ids, f, ensure_ascii=False)
    else:
        print("⚠️ 没有有效的文本块用于Embedding")

def _load_cached_embeddings() -> bool:
    """尝试从磁盘加载缓存。成功返回True。"""
    global _cached_chunk_embs, _cached_chunk_ids
    if not (os.path.exists(EMB_CACHE_FILE) and os.path.exists(EMB_META_FILE)):
        return False
    try:
        print("📥 Loading cached embeddings...")
        _cached_chunk_embs = torch.load(EMB_CACHE_FILE, map_location="cpu")
        with open(EMB_META_FILE, "r", encoding="utf-8") as f:
            _cached_chunk_ids = json.load(f)
        print("✅ Cache loaded")
        return True
    except Exception as e:
        print(f"Cache load failed: {e}")
        _cached_chunk_embs, _cached_chunk_ids = None, None
        return False

/home/oysl/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
## 对用户问题通过DeepSeek-V3生成英文Task Description
import re
from typing import Optional

def generate_query_task(question: str, model_path: str = "") -> Optional[str]:
    """
    使用 DeepSeek-V3 API 生成英文任务描述（解题步骤）。
    输入 question 已经是英文子问题。
    """
    # 英文版提示词 - 经过优化以精准匹配知识库格式
    prompt = f"""You are an Expert in Electrical Engineering Knowledge Retrieval. Your goal is to convert a specific sub-problem into a standardized "Task Description" that PERFECTLY matches the format used in our technical knowledge base.

The retrieval target (Task Description in knowledge base) uses a rigorous, sequential format: "1. Determine [Input 1]. 2. Determine [Input 2]. ... N. Calculate [Formula]. N+1. Output [Result]."

【Target Format Guidelines】
- **Explicit Inputs**: Identify every variable needed for the formula separately.
- **LaTeX Math**: Use standard LaTeX for symbols (e.g., \\sigma_{{20}}, \\theta, \\times). Do NOT use Markdown code blocks or $ delimiters.
- **Step-by-Step Logic**: Logically list variable acquisition steps BEFORE the calculation step.
- **Formula Explicit**: Include the specific formula expression in the "Calculate" step if inferable.

【High-Quality Examples】

Example A (Reactive Compensation):
Input: "Calculate reactive compensation capacity Q_c given active power P and power factors."
Output: 1. Determine minimum load active power P_{{min}}. 2. Obtain tangent of minimum load power factor \\tan\\phi_{{1min}}. 3. Obtain tangent of target power factor \\tan\\phi_{{2}}. 4. Calculate Q_{{Cmin}} \\le P_{{min}} \\times (\\tan\\phi_{{1min}} - \\tan\\phi_{{2}}). 5. Output maximum allowed Q_{{Cmin}}.

Example B (Conductivity Correction):
Input: "Convert the measured conductivity at 26 degrees to standard 20 degrees."
Output: 1. Determine measured volume conductivity \\sigma_\\theta. 2. Determine temperature coefficient b. 3. Measure experimental temperature \\theta. 4. Calculate \\sigma_{{20}} = \\sigma_\\theta \\times [1 - b(\\theta - 20)]. 5. Output corrected conductivity \\sigma_{{20}}.

Example C (Equivalent Salt Deposit Density):
Input: "Calculate ESDD given salinity and washing parameters."
Output: 1. Determine salinity S_s. 2. Measure volume of washing water V. 3. Measure surface area A. 4. Calculate \\text{{ESDD}} = S_s \\times V / A. 5. Output \\text{{ESDD}}.

【Your Task】
Generate the standardized Task Description for:
Problem: "{question}"

Requirements:
1. STRICTLY follow the "1. Determine ... 2. Measure ... 3. Calculate ... 4. Output ..." flow.
2. Break down variables needed for the formula step-by-step.
3. Use raw LaTeX format for ALL variables and formulas (e.g., \\sigma_\\theta, \\times).
4. Do NOT use Markdown delimiters like $ or $$. Write LaTeX directly.
5. Output as a SINGLE PARAGRAPH with numbered steps.

Task Description:"""

    try:
        response = client.chat.completions.create(
            model="deepseek-ai/DeepSeek-V3",
            messages=[
                {"role": "system", "content": "You are an expert in converting EE problems into executable solution steps."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.2,
            max_tokens=600,
        )
        generated_text = response.choices[0].message.content or ""

        # 简单的清理，去除可能的Markdown标记
        clean_text = generated_text.strip()
        clean_text = re.sub(r"^```.*?\n", "", clean_text)
        clean_text = re.sub(r"```$", "", clean_text)
        
        # 如果GPT输出了换行列表，尝试合并为单行（可选，视知识库格式而定，这里保持单行更接近示例）
        if "\n" in clean_text:
             lines = [ln.strip() for ln in clean_text.split('\n') if ln.strip()]
             # 简单的启发式：如果看起来像列表，就用空格连接
             clean_text = " ".join(lines)
        
        return clean_text

    except Exception as e:
        print(f"DeepSeek-V3 生成解题步骤失败: {str(e)}")
        # Fallback to simple query
        return question



In [8]:
def find_top_k_chunks(query_task: str, chunks: List[Dict], model, tokenizer, k: int = 3) -> Tuple[List[Dict], List[float]]:
    """
    使用缓存的chunk embeddings进行快速匹配，返回Top K个candidates及其分数。
    """
    global _cached_chunk_embs, _cached_chunk_ids, _similarity_log_buffer

    # 1. 尝试加载缓存或计算
    if _cached_chunk_embs is None or _cached_chunk_ids is None:
        if not _load_cached_embeddings():
            # 只有当磁盘没有缓存时，才进行预计算
            precompute_chunk_embeddings(chunks, model, tokenizer)

    try:
        # 对查询文本编码（先清理）
        # 【强制使用 CPU】避免 GPU OOM
        query_task_clean = _strip_latex_markdown(query_task)
        query_emb = get_embeddings([query_task_clean], model, tokenizer, device="cpu")[0]  # [D]
        
        # 确保缓存也在CPU上以便计算
        if _cached_chunk_embs.device.type != 'cpu':
             _cached_chunk_embs = _cached_chunk_embs.cpu()
             
        scores = torch.matmul(_cached_chunk_embs, query_emb).tolist()  # [N]

        # 获取 Top K
        top_k_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        
        results = []
        result_scores = []
        
        idx_map = {str(c.get('chunk_id', f'idx_{i}')): i for i, c in enumerate(chunks)}

        _similarity_log_buffer.append(f"Top {k} 检索结果：")
        
        for rank, idx in enumerate(top_k_indices, 1):
            score = scores[idx]
            cid = _cached_chunk_ids[idx]
            
            # Find original chunk
            chunk_idx = idx_map.get(cid, idx)
            chunk = chunks[chunk_idx].copy() 
            
            chunk['_match_score'] = float(score)
            chunk['_match_chunk_id'] = str(cid)
            
            results.append(chunk)
            result_scores.append(score)
            
            log_line = f"  #{rank} [{cid}] sim={score:.4f}"
            print(log_line)
            _similarity_log_buffer.append(log_line)

        return results, result_scores

    except Exception as e:
        err = f"快速匹配出错：{e}, 回退到Top-1旧逻辑返回单元素的list"
        print(err)
        _similarity_log_buffer.append(err)
        return ([chunks[0]], [0.0]) if chunks else ([], [])

In [9]:
## 显示匹配到的公式信息（适配英文Key）
def show_matched_formula_info(chunk: Dict):
    """显示匹配到的公式详细信息（适配新的英文JSON结构）"""
    print("\n🎯 匹配到的公式信息：")
    print("="*60)
    
    # 1. 基础信息
    if 'chunk_id' in chunk:
        print(f"🆔 Chunk ID: {chunk['chunk_id']}")
    
    # 2. 公式
    if 'formula' in chunk and chunk['formula']:
        print(f"📐 Formula: {_strip_latex_markdown(chunk['formula'])}")
        
    # 3. 完整上下文/描述
    if 'full_context' in chunk and chunk['full_context']:
        print(f"📝 Context: {_strip_latex_markdown(chunk['full_context'])[:200]}...")
        
    # 4. 变量定义 (variable_definitions is a list of dicts)
    if 'variable_definitions' in chunk and isinstance(chunk['variable_definitions'], list):
        print("\n📊 Variable Definitions:")
        for var in chunk['variable_definitions']:
            if isinstance(var, dict):
                symbol = _strip_latex_markdown(var.get('symbol', ''))
                name = _strip_latex_markdown(var.get('name', ''))
                meaning = _strip_latex_markdown(var.get('meaning', ''))
                unit = _strip_latex_markdown(var.get('unit', ''))
                
                line = f"  • {symbol}: {name}"
                if meaning:
                    line += f" ({meaning})"
                if unit:
                    line += f" [{unit}]"
                print(line)

    # 5. 关键词
    if 'keywords' in chunk and chunk['keywords']:
        kws = chunk['keywords']
        if isinstance(kws, list):
            print(f"\n🔑 Keywords: {', '.join(kws)}")
        else:
            print(f"\n🔑 Keywords: {kws}")

    # 6. 任务描述
    if 'task_description' in chunk and chunk['task_description']:
        print(f"\n🎯 Task Description: {_strip_latex_markdown(chunk['task_description'])}")
    
    print("="*60)

In [10]:
## 构建调用知识库之后强化Prompt（适配英文Key）
def build_enhanced_prompt(user_question: str, query_task: str, chunk: Dict) -> str:
    """
    构造强化prompt (Legacy function, mainly used by single search, but updated for compatibility)
    """
    # 格式化chunk信息
    chunk_info = []
    
    # 处理 variable_definitions
    if 'variable_definitions' in chunk and isinstance(chunk['variable_definitions'], list):
        vars_str = []
        for v in chunk['variable_definitions']:
            if isinstance(v, dict):
                s = f"{v.get('symbol','')} : {v.get('name','')} (Unit: {v.get('unit','')})"
                vars_str.append(s)
        if vars_str:
            chunk_info.append("Variable Definitions:\n  - " + "\n  - ".join(vars_str))
            
    # 处理其他字段
    for key in ['formula', 'full_context', 'task_description', 'keywords']:
        if key in chunk and chunk[key]:
            val = chunk[key]
            if isinstance(val, list):
                val = ", ".join(str(x) for x in val)
            chunk_info.append(f"{key}: {_strip_latex_markdown(str(val))}")

    chunk_info_str = "\n".join(chunk_info)
    
    prompt = f"""Please answer the question following this logic:
(1) Understand User Requirement:
{_strip_latex_markdown(user_question)}

(2) Execution Steps:
{_strip_latex_markdown(query_task)}

(3) Reference Knowledge:
{chunk_info_str}

Requirements:
- Must use formulas and variables from Reference Knowledge.
- Steps must correspond to the task description.
- Final result must have units consistent with definitions.
- Use LaTeX format for all variables and formulas (e.g., \\sigma_\\theta, \\times, \\sqrt{{...}}).

Answer:"""
    return prompt


In [11]:
## 构建综合Prompt工具函数
def build_comprehensive_prompt(
    user_question_cn: str,
    user_question_en: str,
    sub_problems: List[str], 
    task_descriptions: List[str], 
    matched_chunks: List[Dict]
) -> str:
    """
    构造综合的强化prompt：
    - 主问题：中文 + 英文
    - 子问题：英文
    - 知识块：英文 (适配新Key)
    """
    
    n = len(sub_problems)
    # 补齐
    if len(task_descriptions) < n:
        task_descriptions += [_strip_latex_markdown(sp) for sp in sub_problems[len(task_descriptions):]]
    if len(matched_chunks) < n:
        matched_chunks += [{} for _ in range(n - len(matched_chunks))]
    
    # 1. 构建子问题解析部分 (English)
    sub_problems_section = []
    for i in range(n):
        sub_prob = sub_problems[i]
        task_desc = task_descriptions[i]
        sub_problems_section.append(f"""
[Sub-problem {i+1}]
Description: {_strip_latex_markdown(sub_prob)}
Plan: {_strip_latex_markdown(task_desc)}""")
    
    sub_problems_str = "\n".join(sub_problems_section)
    
    # 2. 构建所有相关知识部分 (English Keys)
    all_knowledge = []
    for i in range(n):
        chunk = matched_chunks[i] if i < len(matched_chunks) else {}
        chunk_info = []
        
        if isinstance(chunk, dict):
            # 显式添加 chunk_id
            if 'chunk_id' in chunk:
                chunk_info.append(f"chunk_id: {chunk['chunk_id']}")

            # 处理 variable_definitions
            if 'variable_definitions' in chunk and isinstance(chunk['variable_definitions'], list):
                vars_list = []
                for v in chunk['variable_definitions']:
                    if isinstance(v, dict):
                        s = f"{v.get('symbol','')} : {v.get('name','')} (Meaning: {v.get('meaning','')}, Unit: {v.get('unit','')})"
                        vars_list.append(s)
                if vars_list:
                    chunk_info.append(f"variable_definitions:\n    - " + "\n    - ".join(vars_list))
            
            # 处理其他常见字段
            for key in ['formula', 'keywords', 'task_description', 'full_context']:
                if key in chunk and chunk[key]:
                    val = chunk[key]
                    if isinstance(val, list):
                        val = ", ".join(str(x) for x in val)
                    chunk_info.append(f"{key}: {_strip_latex_markdown(str(val))}")
        
        knowledge_block = f"""
[Knowledge Block {i+1}] (For Sub-problem {i+1})
{chr(10).join(chunk_info)}"""
        all_knowledge.append(knowledge_block)
    
    all_knowledge_str = "\n".join(all_knowledge)
    
    # 3. 构建最终的综合prompt
    comprehensive_prompt = f"""Please strictly follow the logic chain below to solve this complex engineering problem:

.
【Original Complex Problem】
(Chinese): {_strip_latex_markdown(user_question_cn)}
(English Translation): {_strip_latex_markdown(user_question_en)}

.
【Problem Decomposition】
The problem has been decomposed into {n} related sub-problems:
{sub_problems_str}

.
【Reference Knowledge Base】
{all_knowledge_str}

.
【Solution Requirements】
1. Solve sequentially following the sub-problem logic.
2. Use ONLY formulas and variable definitions from the Reference Knowledge Base for each sub-problem.
3. The output of the previous sub-problem serves as input for the next.
4. Final result must include units consistent with definitions.
5. Provide complete derivation process and final answer.
6. Use LaTeX format for all variables and formulas in the final answer (e.g., \\sigma_\\theta, \\times, \\sqrt{{...}}).

Answer:"""
    return comprehensive_prompt


### 4. Agent loop and revision
This section implements the full agent with **five roles**: Planner, Retriever (no re-rank), Solver, Critic, and Editor. The loop iterates up to 3 rounds — the Critic audits each answer for consistency/completeness, and if the score falls below the threshold (0.85), the system triggers re-retrieval or re-planning for the next round.

**Core roles:**
- **Planner**: Translates CN→EN, then decomposes the problem into sub-problems via DeepSeek-V3.
- **Retriever** *(noRerank)*: Generates task descriptions and retrieves top-K chunks by cosine similarity — **no GPT re-ranking**.
- **Solver**: Builds an augmented prompt from matched chunks and generates a candidate solution via DeepSeek-V3.
- **Critic**: Audits the answer for formula consistency, unit correctness, and completeness.
- **Editor**: Adjusts strategy (e.g., restricts to two formulas, reorders steps) for the next round.

---

This section connects Planner、Retriever（无重排序）、Solver 和 Critic 形成闭环。Agent 最多迭代 3 轮：若 Critic 评分低于阈值，则触发重新检索或重新规划，进入下一轮。唯一的消融点在于 Retriever 阶段没有 GPT 重排序。


In [12]:
# Agent 配置
import os, time, json

AGENT_CFG = {
    "max_rounds": 3,                 # 最多迭代轮数
    "two_formula_mode": False,       # True 则强制仅两条公式/两步子问题
    "min_score": 0.85,               # 评审者通过阈值（0-1），0.85及以上通过
    "min_similarity": 0.75,          # 子问题匹配相似度阈值
    "save": {
        "prompt_path": "full_prompt_agent/default.txt", # 默认路径，会在运行时被覆盖
        "log_dir": "logs"
    }
}

def _now_tag():
    return time.strftime("%Y%m%d-%H%M%S", time.localtime())

In [13]:
# 角色实现：Planner / Retriever / Solver / Critic / Editor
import re, json
from typing import Tuple, List, Dict, Any
import numpy as np

def _parse_json_loose(s: str) -> Dict[str, Any]:
    try:
        return json.loads(s)
    except Exception:
        s2 = re.sub(r"^```(json)?|```$", "", s.strip(), flags=re.MULTILINE)
        try:
            return json.loads(s2)
        except Exception:
            return {}

def role_planner(user_question: str, two_formula_mode: bool=False) -> Tuple[List[str], str]:
    # 返回 (sub_problems, english_question)
    subs, eng_q = decompose_complex_problem(user_question)
    if not subs:
        subs = [eng_q]
    if two_formula_mode and len(subs) > 2:
        subs = subs[:2]
    return subs, eng_q

def select_top_chunk_by_similarity(sub_prob: str, candidates: List[Dict]) -> Dict:
    """
    noGPT 模式：不调用额外 API 精排，直接返回向量检索的 Top-1 结果。
    """
    if not candidates:
        return {}
    return candidates[0]

def role_retriever(sub_problems: List[str]) -> Tuple[List[str], List[Dict], List[float]]:
    """
    noGPT Retriever:
    1. 为每个子问题检索Top-3
    2. 直接选择相似度最高的Top-1，不做额外精排
    3. 返回选中chunk以及 Top-1 Similarity Scores (用于Threshold Check)
    """
    task_descs, matched = [], []
    top1_scores = []  # 用于 "highest matching rate formula" 检查
    
    for sp in sub_problems:
        # sp is English now
        t = generate_query_task(sp) or _strip_latex_markdown(sp)
        task_descs.append(_strip_latex_markdown(t))
        
        # 1. Get Top 3 (Retrieval)
        cands, scores = find_top_k_chunks(task_descs[-1], all_chunks, model, tokenizer, k=3)
        
        # 记录每组这一轮里 Top 1 的分数 (作为 "highest matching rate" 依据)
        top1_scores.append(scores[0] if scores else 0.0)
        
        # 2. noGPT: 直接取相似度最高的候选
        best = cands[0] if cands else {}
        matched.append(best)
        
    return task_descs, matched, top1_scores

# Solver, Critic, Editor 保持不变
def role_solver(user_question_cn: str, user_question_en: str, sub_problems: List[str], task_descs: List[str], matched_chunks: List[Dict], two_formula_mode: bool=False) -> Tuple[str, str]:
    # 构建综合prompt + Agent约束
    base_prompt = build_comprehensive_prompt(user_question_cn, user_question_en, sub_problems, task_descs, matched_chunks)
    constraints = []
    if two_formula_mode:
        constraints.append("Strictly use no more than two formulas. Prioritize the two retrieved knowledge blocks.")
    constraints.append("All derivation steps must cite variable definitions and units from the Knowledge Blocks. Do not introduce unknown formulas.")
    final_prompt = base_prompt + "\n【Agent Constraints】\n- " + "\n- ".join(constraints)
    # 使用GPT求解
    response = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3",
        messages=[
            {"role": "system", "content": "You are a rigorous Engineering Field Solver. Strictly follow the provided Knowledge and Constraints."},
            {"role": "user", "content": final_prompt}
        ],
        temperature=0.2, max_tokens=1800,
    )
    answer = response.choices[0].message.content or ""
    return final_prompt, answer

def role_critic(user_question: str, matched_chunks: List[Dict], answer: str, two_formula_mode: bool=False) -> Dict[str, Any]:
    # 构造评审提示，要求输出JSON
    kb_brief = []
    for i, ch in enumerate(matched_chunks, 1):
        # English keys
        kb_brief.append(f"[Block {i}] Formula: {ch.get('formula','')}")
        
    critic_prompt = f"""
You are a strict Technical Critic. Evaluate the solution below for consistency and constraints. Output JSON.

【Original Question】
{_strip_latex_markdown(user_question)}

【Knowledge Used】
{chr(10).join(_strip_latex_markdown(x) for x in kb_brief)}

【Model Solution】
{_strip_latex_markdown(answer)}

【Evaluation Criteria】
- Consistency: Uses ONLY definitions/formulas/units from Knowledge Blocks?
- Completeness: Are steps closed-loop? Are units correct?
- Constraints: {'Only 2 formulas allowed' if two_formula_mode else 'Multi-step allowed'}

【Output JSON Fields (Strict JSON only)】
{{
  "score": 0.0-1.0,
  "needs_revision": true/false,
  "issues": ["issue1", "issue2"],
  "suggestions": ["suggestion1", "suggestion2"]
}}
"""
    response = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3",
        messages=[
            {"role": "system", "content": "You are a strict Technical Critic. Output must be JSON."},
            {"role": "user", "content": critic_prompt}
        ],
        temperature=0.0, max_tokens=600,
    )
    raw = response.choices[0].message.content or ""
    data = _parse_json_loose(raw)
    if not isinstance(data, dict):
        data = {}
    data.setdefault("score", 0.0)
    data.setdefault("needs_revision", True)
    if "issues" not in data:
        data["issues"] = ["Failed to parse JSON"]
    data.setdefault("suggestions", [])
    return data

def role_editor(sub_problems: List[str], feedback: Dict[str, Any], cfg: Dict[str, Any]) -> List[str]:
    sp = list(sub_problems)
    if cfg.get("two_formula_mode", False):
        sp = sp[:2] if len(sp) >= 2 else sp
    if feedback.get("needs_revision", True):
        # 可以在这里根据feedback调整，当前保持简单逻辑
        pass
    return sp

In [14]:
# 编排器：多轮对话式Agent主流程
from typing import NamedTuple

class AgentResult(NamedTuple):
    answer: str
    final_prompt: str
    score: float
    rounds: int
    feedbacks: List[Dict[str, Any]]

def run_agent(user_question: str, cfg: Dict[str, Any]) -> AgentResult:
    # 0. 自动加载必要的全局资源 (模型 & 知识库)
    global model, tokenizer, all_chunks
    
    if 'model' not in globals() or 'tokenizer' not in globals():
        print("🔧 检测到模型未加载，正在初始化 Embedding 模型...")
        model, tokenizer = load_embedding_components()
        print("✅ 模型加载完成")

    if 'all_chunks' not in globals() or not all_chunks:
        print("📊 检测到知识库未加载，正在加载默认路径数据...")
        # 尝试使用默认路径
        default_path = "jsons_EE_Eng"
        all_chunks = load_jsonl_files(default_path)

    # 确保目录存在 (使用传入的cfg配置)
    os.makedirs(os.path.dirname(cfg["save"]["prompt_path"]), exist_ok=True)
    os.makedirs(cfg["save"]["log_dir"], exist_ok=True)

    logs = []
    feedbacks: List[Dict[str, Any]] = []
    two_formula_mode = cfg.get("two_formula_mode", False)
    max_rounds = int(cfg.get("max_rounds", 2))
    min_score = float(cfg.get("min_score", 0.8))
    sim_threshold = float(cfg.get("min_similarity", 0.75))
    
    answer = ""
    final_prompt = ""
    sub_problems: List[str] = []
    english_question = ""
    
    best_score = -1.0
    best_answer = ""
    best_prompt = ""
    best_feedback: Dict[str, Any] = {}
    best_round = 0
    rd = 1
    
    while rd <= max_rounds:
        logs.append(f"=== Round {rd} ===")
        print(f"\n🟢 开始第 {rd} 轮 ...")
        
        # 1) Planner (Returns subs + eng_q)
        if rd == 1:
            # 第一轮调用Planner进行翻译和拆解
            sub_problems, english_question = role_planner(user_question, two_formula_mode=two_formula_mode)
        else:
            # 后续轮次可能由Editor修改了sub_problems，但english_question不变
            pass
            
        logs.append(f"Planner: Sub-problems count={len(sub_problems)}")
        
        # 2) Retriever (Updated: Return matched chunks AND sim scores)
        # matched 已经是 GPT筛选后的 top-1 chunk list
        task_descs, matched, sim_scores = role_retriever(sub_problems)
        
        logs.append("Retriever: Matched={} | Top-1 Sim={}".format(len(matched), [f"{s:.4f}" for s in sim_scores]))
        
        # 3) Strict Threshold Check (放弃机制：如果相似度不够，直接重开)
        if any(s < sim_threshold for s in sim_scores):
            msg = f"⚠️ 本轮检索相似度过低 (Threshold={sim_threshold:.2f}). Scores: {[f'{s:.2f}' for s in sim_scores]}. 放弃本轮结果，尝试调整..."
            print(msg)
            logs.append(msg)
            
            # 手动触发 Editor，传入模拟的feedback让它去修改子问题
            mock_feedback = {
                 "needs_revision": True,
                 "issues": ["Retrieval similarity too low. Rephrase sub-problems."]
            }
            logs.append("✏️ 因相似度不足触发 Editor 修改计划...")
            sub_problems = role_editor(sub_problems, mock_feedback, cfg)
            
            rd += 1
            continue  # <--- 关键修改：直接跳过 Solver 和 Critic，进入下一轮
        
        # 4) Solver (Pass both CN and EN questions)
        final_prompt, answer = role_solver(user_question, english_question, sub_problems, task_descs, matched, two_formula_mode=two_formula_mode)
        logs.append(f"Solver: Answer length={len(answer)}")
        
        # 5) Critic
        fb = role_critic(user_question, matched, answer, two_formula_mode=two_formula_mode)
        feedbacks.append(fb)
        score = float(fb.get("score", 0.0))
        logs.append(f"Critic: score={score}, needs_revision={fb.get('needs_revision', True)}")
        print(f"📊 本轮 Critic 打分: {score}")

        # 筛选逻辑：保留打分最高的一轮（这里依然使用 Critic Score 作为最终质量标准）
        # 前提是该轮已经通过了 Similarity Threshold 检查
        if score > best_score:
            best_score = score
            best_answer = answer
            best_prompt = final_prompt
            best_feedback = fb
            best_round = rd
            print(f"🌟 新的高分轮次! (Score: {score})")
            
        # 6) Stop or Edit
        if score >= min_score or not fb.get("needs_revision", True):
            logs.append("⏹ Threshold met. Stopping.")
            print("✅ 达到分数阈值，提前结束循环。")
            break
        else:
            logs.append("✏️ Threshold not met. Editing...")
            sub_problems = role_editor(sub_problems, fb, cfg)
            rd += 1
    
    # 最终结果输出
    if best_score >= 0:
        chosen_answer = best_answer
        chosen_prompt = best_prompt
        chosen_feedback = best_feedback
        print(f"\n🏆 最终选择第 {best_round} 轮结果 (Critic Score: {best_score})")
    else:
        # 兜底：如果所有轮都失败（例如都低于Sim阈值），则返回最后一次的尝试（可能是空的或质量差的）
        chosen_answer = answer
        chosen_prompt = final_prompt
        chosen_feedback = feedbacks[-1] if feedbacks else {}
        print("\n⚠️ 未能找到满意的结果，返回最后一次尝试。")
    
    logs.append(f"Best Score: {best_score} (Round {best_round})")
    
    ts = _now_tag()
    log_path = os.path.join(cfg["save"]["log_dir"], f"agent_run_{ts}.txt")
    with open(log_path, "w", encoding="utf-8") as f:
        f.write("\n".join(logs))
        f.write("\n\nBest Round Feedback:\n")
        try:
            f.write(json.dumps(chosen_feedback, ensure_ascii=False, indent=2))
        except Exception:
            f.write(str(chosen_feedback))
            
    with open(cfg["save"]["prompt_path"], "w", encoding="utf-8") as f:
        f.write(chosen_prompt)
        
    print(f"Log saved: {log_path}")
    print(f"Prompt saved: {cfg['save']['prompt_path']}")
    return AgentResult(answer=chosen_answer, final_prompt=chosen_prompt, score=best_score, rounds=len(feedbacks), feedbacks=feedbacks)

### 5. Batch generation for comparison
This section runs the notebook on a question range and writes one output file per item — with the full agent loop but without GPT re-ranking.

---

这部分会按题号范围批量运行 notebook，每道题以完整 Agent 闭环（但无 GPT 重排序）保存一个输出文件。


In [ ]:
# ================= 批量运行模式 =================
# 配置区域 - 在这里修改文件名和范围
BATCH_INPUT_FILE = "EE_Question/EE_Question_Batch.txt"  # 问题输入文件
OUTPUT_DIR = "Prompt_noRerank"       # 结果输出文件夹 (相对于当前目录)

# ***在此处设置处理范围***
START_INDEX = 1        # 起始编号 (例如 1 或 101)
END_INDEX = 100        # 结束编号 (例如 100 或 200)
# 当你想跑 101-200 时，只需修改上面: START_INDEX=101, END_INDEX=200

# 更新Agent全局默认配置
AGENT_CFG["two_formula_mode"] = False       
AGENT_CFG["save"]["log_dir"] = "logs"

import os
import traceback

# 确保输出目录存在
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. 读取问题
questions = []
if os.path.exists(BATCH_INPUT_FILE):
    with open(BATCH_INPUT_FILE, "r", encoding="utf-8") as f:
        # 简单处理：过滤掉注释行和空行
        for line in f:
            ln = line.strip()
            if ln and not ln.startswith("#"):
                questions.append(ln)
    print(f"📖 从 {BATCH_INPUT_FILE} 加载了 {len(questions)} 个问题。")
else:
    print(f"⚠️ 未找到文件 {BATCH_INPUT_FILE}，请先在 {BATCH_INPUT_FILE} 中填入问题。")
    questions = []

# 2. 批量执行
# 切片获取指定范围的问题. Python list索引从0开始, 所以要减1
# 如果 START_INDEX=1, 切片 [0:100] -> questions[0]...questions[99] -> 对应 1..100
target_questions = questions[START_INDEX-1 : END_INDEX]

if not target_questions:
    print(f"⚠️ 警告: 指定范围 {START_INDEX}-{END_INDEX} 没有取到问题 (总问题数: {len(questions)})")
else:
    print(f"🎯 准备处理序号 {START_INDEX} 到 {END_INDEX} 的问题，共 {len(target_questions)} 个。")

    for i, q in enumerate(target_questions):
        current_idx = START_INDEX + i
        
        # 构造输出文件名: Question101.txt ... 
        filename = f"Question{current_idx}.txt"
        output_path = os.path.join(OUTPUT_DIR, filename)
        
        print(f"\n" + "="*50)
        print(f"🚀 [Processing Item {current_idx}] (Batch Progress: {i+1}/{len(target_questions)})")
        print(f"📝 Question: {q[:50]}...")
        # print(f"Checking output path: {output_path}")
        
        # 动态更新保存路径
        AGENT_CFG["save"]["prompt_path"] = output_path
        
        try:
            # 执行Agent
            res = run_agent(q, AGENT_CFG)
            print(f"✅ Item {current_idx} Finished. Score: {res.score}. Output -> {output_path}")
        except Exception as e:
            print(f"❌ Item {current_idx} Failed: {str(e)}")
            traceback.print_exc()
            # 遇到错误继续下一个
            continue

    print(f"\n🎉 范围 {START_INDEX}-{END_INDEX} 批量任务完成！结果保存在 {OUTPUT_DIR}/ 目录下。")

📖 从 /data1/oysl/RAG/EE_Question/EE_Question_Batch.txt 加载了 288 个问题。
🎯 准备处理序号 1 到 100 的问题，共 100 个。

🚀 [Processing Item 1] (Batch Progress: 1/100)
📝 Question: 1. 一台间接冷却燃气轮发电机，初级冷却介质入口温度t=5°C。查表得其标准温升限值为80K。请计算...
🔧 检测到模型未加载，正在初始化 Embedding 模型...
Loading embedding model from /data1/oysl/RAG/Qwen3-Embedding...
✅ 模型加载完成

🟢 开始第 1 轮 ...
🌐 正在将问题翻译为英文...
📝 英文问题: 1. An indirectly cooled gas turbine generator has an inlet temperature of the primary cooling medium at t=5°C. According to the table, its standard temperature rise limit is 80K. Calculate the allowable temperature rise limit of the winding under this cooling medium temperature (i.e., the standard limit plus the adjustment ΔT).
🔧 正在调用DeepSeek-V3拆解复合题目(英文版)...
✅ DeepSeek-V3拆解完成
📋 拆解结果（原始）：
【Variable Extraction】
Known Variables: 
- Inlet temperature of primary cooling medium, t = 5°C
- Standard temperature rise limit, ΔT_{\text{standard}} = 80K

Target Variable: 
- Allowable temperature rise limit of the winding, ΔT_{\text{allowable}}

【Fo

KeyboardInterrupt: 